In [1]:
from tree.node import TerminalNode
import utils.constants as constants
from tree.tree import ReadTree
from tree.node import RootNode, SemanticNode, SyntacticNode
from rouge_score import rouge_scorer
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import colorsys
from collections import deque
import re
import pandas as pd
import ast
scorer = rouge_scorer.RougeScorer(['rougeL', 'rouge1'], use_stemmer=True)

06/18/2025 12:15:06 - INFO - 	 Using default tokenizer.


In [2]:
import pandas as pd
df = pd.read_csv('/homes/lst20/fyp/fyp_resources/LexEval-main/analysis/metric_comparison_merged.csv')
df.columns

Index(['possible_answers', 'base_rag', 'root_prompt', 'does rag imply ref',
       'mnli_probs_base_rag_to_ref', 'mnli_label_base_rag_to_ref', 'rag_f1_1',
       'rag_f1_L', 'rag_rec_1', 'rag_rec_L', 'rag_prec_1', 'rag_prec_L',
       'rag_em'],
      dtype='object')

In [3]:
import ast
import json
import re
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

import ast

def unexplode_possible_ans(df):
    # Step 2: Drop the 'processed_possible_answer' column if it exists
    if 'processed_possible_answer' in df.columns:
        df = df.drop(columns=['processed_possible_answer'])

    # Step 3: For each question_id, keep the row with the max prob
    df = df.loc[df.groupby('question_id')['mnli_probs_base_rag_to_ref'].idxmax()]

    return df.reset_index(drop=True)


    
def add_rouge_scores(df):
    # Initialize columns for storing the best scores
    metric_keys = [
        'rag_f1_1', 'rag_f1_L', 
        'rag_rec_1', 'rag_rec_L', 
        'rag_prec_1', 'rag_prec_L',
        'rag_em'
    ]
    for key in metric_keys:
        df[key] = 0.0

    # Process each row
    for idx, row in df.iterrows():
        possible_answers = row['possible_answers']
        rag_response = str(row['base_rag']).lower()
        rag_response = re.sub(r'[^\w\s]', '', rag_response)

        try:
            parsed = json.loads(possible_answers)
            possible_answers = parsed if isinstance(parsed, list) else [parsed]
        except json.JSONDecodeError:
            possible_answers = ast.literal_eval(possible_answers)
        
        rag_found_match = False
        best = {k: -1.0 for k in metric_keys}

        for exp in possible_answers:
            exp_low = str(exp).lower()
            exp_low = re.sub(r'[^\w\s]', '', exp_low)

            if exp_low in rag_response:
                rag_found_match = True

            scores = scorer.score(exp_low, rag_response)
            metrics = {
                'rag_f1_1': scores['rouge1'].fmeasure,
                'rag_f1_L': scores['rougeL'].fmeasure,
                'rag_rec_1': scores['rouge1'].recall,
                'rag_rec_L': scores['rougeL'].recall,
                'rag_prec_1': scores['rouge1'].precision,
                'rag_prec_L': scores['rougeL'].precision,
                'rag_em': float(rag_found_match)
            }

            for k, v in metrics.items():
                if v > best[k]:
                    best[k] = v

        # Update row in DataFrame
        for k in metric_keys:
            df.at[idx, k] = best[k]

    return df

06/18/2025 12:15:06 - INFO - 	 Using default tokenizer.


In [4]:
from sklearn.metrics import roc_auc_score
    
from sklearn.metrics import (
    roc_auc_score, precision_score,
    recall_score, f1_score, accuracy_score
)

def compute_roc_aucs(df):
    y_true = df['does rag imply ref'].astype(int)

    # Columns with scores (probabilities or continuous predictions)
    score_cols = [
        'rag_em',
        'rag_f1_1',
        'rag_prec_1',
        'rag_rec_1',
        'mnli_label_base_rag_to_ref',
        'mnli_probs_base_rag_to_ref'
    ]
    
    aucs = {}
    for col in score_cols:
        try:
            aucs[f'{col}_roc_auc'] = roc_auc_score(y_true, df[col])
        except ValueError as e:
            aucs[f'{col}_roc_auc'] = f'Error: {e}'

    # Binary predictions for mnli_label_base_rag_to_ref
    y_pred = df['mnli_label_base_rag_to_ref'].astype(int)

    # Compute classification metrics
    try:
        aucs['mnli_precision'] = precision_score(y_true, y_pred)
        aucs['mnli_recall'] = recall_score(y_true, y_pred)
        aucs['mnli_f1'] = f1_score(y_true, y_pred)
        aucs['mnli_accuracy'] = accuracy_score(y_true, y_pred)
    except ValueError as e:
        # In case of empty or invalid predictions
        aucs['mnli_error'] = f'Error: {e}'

    return aucs


In [5]:
add_rouge_scores(df)
new_aucs = compute_roc_aucs(df)

# Print AUCs
for metric, score in new_aucs.items():
    print(f"{metric}: {score}")

rag_em_roc_auc: 0.8218476438815422
rag_f1_1_roc_auc: 0.8666418327435276
rag_prec_1_roc_auc: 0.8658968150493573
rag_rec_1_roc_auc: 0.8532315142484633
mnli_label_base_rag_to_ref_roc_auc: 0.874743900167629
mnli_probs_base_rag_to_ref_roc_auc: 0.9139504563233377
mnli_precision: 0.9186046511627907
mnli_recall: 0.8681318681318682
mnli_f1: 0.8926553672316384
mnli_accuracy: 0.8733333333333333
